## Attention weighting without forced per-sequence winners

The current implementation normalizes attention separately within every rollout:

$$
w_{b,t}=\frac{s_{b,t}}{\operatorname{mean}_{u\in\mathrm{CoT}_b}s_{b,u}}
$$

where $s_{b,t}$ is the mean attention from the answer tokens in rollout $b$ to CoT token $t$. This forces the average token weight in every rollout to be 1. Consequently, even when the answer barely attends to a rollout's entire CoT, that rollout still gets relatively high-attention "winner" tokens.

Batch-wide normalization of raw per-token scores would retain differences between rollouts, but introduces a length bias: attention sums to 1 for each query, so longer CoTs naturally receive smaller attention per token.

A cleaner design separates sequence-level usefulness from within-sequence token selection. First compute the total answer-attention mass assigned to the CoT:

$$
u_b=\sum_{t\in\mathrm{CoT}_b}s_{b,t}
$$

Then combine a sequence-level gate with the relative token score:

$$
w_{b,t}=
\frac{u_b}{\operatorname{EMA}(u)}
\cdot
\frac{s_{b,t}}{\operatorname{mean}_{v\in\mathrm{CoT}_b}s_{b,v}}
$$

Here $u_b/\operatorname{EMA}(u)$ suppresses an entire weakly attended CoT, while the second factor still allocates more credit to its relatively important tokens. A running EMA, or a baseline within the same GRPO prompt group, is preferable to an arbitrary training-batch mean because it is less sensitive to batch composition.